# 19.18 三個成品要如何比較才有意義？


MoE 助理能處理有限圖片與聲音，Looped 解題器只處理明確文字語法，機率判斷器回機率判斷。如果把各自最拿手的測試分數排成一欄，就沒有回答共同問題。比較前先問用途：是選一個合適成品、檢查Auto取捨，還是研究循環架構的因果效果？

選成品時，先並排輸入、支援輸出、能力範圍、權重大小與使用成本；三份結果來自不同日期與不同資料，要一起註明。這是用途導覽，不能讀成架構勝負。共同客服場景能讓讀者理解如何合作，也不能抹平任務差異。

| 項目 | MoE 助理 | Looped 解題器 | 機率判斷器 |
| --- | --- | --- | --- |
| 輸入 | 文字與對話歷史；商品圖與兩個格位；字卡與讀字框；錄音 | 兩盒追蹤或三數加法的固定語法字串；輪數 | 固定語法的狀態；問題（choice、info、rubric）；候選 |
| 支援輸出 | 文字回答；計算器請求 | 0–18 的答案類別與實際輪數 | 各候選機率、選中候選、集中度 |
| 能力範圍 | 有限客服文字、三類服飾、12 個字、三種語音意圖；未通過全部判準（[19.12](https://birdhackor.github.io/tiny-perceptron-vlm/19.12.html)） | 1–4 個事件的追蹤通過主要門檻；加法 2/42（[19.15](https://birdhackor.github.io/tiny-perceptron-vlm/19.15.html)） | 主要測試題全對；換措辭後出現高信心錯誤（[19.16](https://birdhackor.github.io/tiny-perceptron-vlm/19.16.html)） |
| 參數與權重檔 | 5,447,107 個；`model.safetensors` 22,366,908 bytes | 170,003 個；`model.pt` 685,437 bytes | 84,545 個；`model.pt` 346,625 bytes |
| CPU 使用成本 | 沒有正式量測；公開示範每次執行 `chat.py`，整個命令約 4–9 秒 | 直接跑 medium（3 輪）平均約 1.30 毫秒 | Auto 流程中的兩次判斷合計平均約 2.9 毫秒 |
| 結果出處 | [v2-final-public-results.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/v2-final-public-results.json)，2026-10-06 | [final-report-v1.json](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/lesson-products/results/final-report-v1.json)，2026-10-16 | 同左 |

MoE 的執行秒數來自[公開 CPU 執行紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/infrastructure/v2-public-cpu-smoke-actual-review.json)：`invocations` 裡呼叫 `chat.py` 的 8 筆 `elapsed_seconds`，是整個命令從啟動到結束的時間，紀錄沒有拆開下載、載入與生成各花多少；另兩欄是下面同一份計時的函式內時間，不能和它直接比快慢。

Auto的比較則有共同測試題：同一凍結解題器、相同解題題目與讀出方式，依次測固定low、medium、high及規則Auto。每題保留答案、輪數、判斷器的判斷、時間及是否受硬上限限制。medium必須在表裡，因為 medium 檔位可能才是最好的固定選擇。平均成本相同時比較正確率，或達同樣品質時比較成本，才知道選擇器是否值得。

回看正確答案、替每題挑最便宜又答對的檔位，得到的是**離線理想上界**（oracle，[7.25](https://birdhackor.github.io/tiny-perceptron-vlm/7.25.html)）：它只屬於這份固定執行紀錄。全部檔都錯的題仍算錯；若還要找出最佳檔，搜尋成本也不是免費。它可以提示Auto還有改善空間，不能和實際可用的決策規則混列成部署成果。

研究循環本身，需要新訓練的匹配 Dense 對照（control）：資料、起點規則、工作量及選權重方式盡量對齊，說明對照的是參數還是計算。MoE 助理的訓練歷史不同，不能拿它來隔離共用區塊的效果。[14.15](https://birdhackor.github.io/tiny-perceptron-vlm/14.15.html)的機制實驗與[19.15](https://birdhackor.github.io/tiny-perceptron-vlm/19.15.html)的產品配方各有自己的結果，不混在同一張表。

本次完整流程用同一份2998題（2956追蹤＋42加法）比較。判斷器回 `unsupported` 或 `clarification_needed` 的題目也留在分母並算答錯；本次 2998 題都被判斷器接受。每列都先作兩次判斷，再由解題器解題。

| 模式 | 答對／2998 | 正確率 | 平均解題輪數 | CPU流程平均毫秒 |
| --- | --- | --- | --- | --- |
| low | 1598 | 53.30% | 1 | 3.63 |
| medium | 2927 | 97.63% | 3 | 4.32 |
| high | 2930 | 97.73% | 5 | 4.99 |
| Auto | 2926 | 97.60% | 2.972 | 4.37 |

品質與平均輪數用全部2998題；時間另依固定ID抽24題，每題重複3次，共72次／模式，先各暖機3次。環境是CPU、float32、2執行緒、batch1，模型已先載入。計時從 `auto_solve` 通過輸入格式檢查之後開始，包含兩次判斷與解題，不含模型載入，也不含網路傳輸。這是函式內等待，不能冒充所有裝置的完整服務延遲。

判斷器兩次平均約2.9毫秒；直接呼叫解題器的medium約1.30毫秒。Auto雖少用約0.93%的解題輪數，本次流程平均時間並未比medium低。少量時間差會受系統波動影響，實驗紀錄保留原始次數與範圍，不把它當可靠尾端延遲保證。記憶體仍要分權重與執行峰值；輪數不是秒數替代品。

練習：Auto少用一輪，卻多跑一個判斷器，總時間可能更長嗎？可能。把判斷器的時間加回來，再看符合實際限制的品質／成本表；別只報對方案有利的代理量。
